# ETF 모의투자 시뮬레이션

이 노트북은 Optuna 결과 Excel에서 config를 읽고, 사용자가 지정한 기간 동안 모의투자 예측을 반복합니다.

구조:

1. `experiment_summary_holdout.xlsx`에서 config 1개 선택
   - 우선 테스트용으로 `row_idx=0` 사용
   - 나중에는 `test_precision` 기준 정렬 후 0번 row를 가져오면 됨

2. 사용자가 지정한 기간의 각 거래일에 대해:
   - 기준일의 전 거래일을 `signal_date`로 사용
   - `signal_date`까지의 데이터만 사용
   - VIF / best lag / RF permutation importance / top_n feature selection
   - 선택 변수로 모델 학습
   - `signal_date` 기준 상승 확률 예측
   - 실제 `n_days` 뒤 결과와 비교

주의:

- 당일 데이터는 사용하지 않음
- 기준일 D의 예측은 D의 전 거래일까지만 보고 만든 신호임


In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import FinanceDataReader as fdr

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
    confusion_matrix
)
from sklearn.inspection import permutation_importance
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler

# =========================================================
# 기본 설정
# =========================================================

### 데이터 티커 및 기간 설정
ETF_CODE = "SMH"
START_DATE = "2020-01-01"
END_DATE = None

### Target 설정
N_DAYS = 5                                  # N일 후의 상승/하락 예측
THRESHOLD = 0.01                            # 상승/하락 판단 기준 (예: 0.05는 5% 상승/하락)
VALID_MONTHS = 1                            # 검증 데이터 기간 (개월 단위, 예: 1은 최근 1개월)


### 변수 선택시 
VIF_THRESHOLD = 10                          # 변수 선택시 VIF 기준 (예: 10 이상인 변수 제거)
LAG_SEARCH_YEARS = 1                        # 변수별 최적 lag 탐색시 사용할 최근 데이터 기간 (년 단위)
LAG_DAYS = [1, 3, 5, 10, 20, 40, 60, 120]   # 변수별 최적 lag 탐색시 사용할 일수 (예: 1, 3, 5, 10, 20, 40, 60, 120일)

### RF 설정 (변수 중요도 파악을 위한 과적합용 모델)
RANDOM_STATE = 42
N_RF_RUNS = 3
N_REPEATS = 10
TOP_N = 30 # 상위 N개 변수 선택
PRED_THRESHOLD = 0.5


# 외부 지표
EXTERNAL_TICKERS = {
    "QQQ": "QQQ",
    "SPY": "SPY",
    "SOXX": "SOXX",
    "NVDA": "NVDA",
    "TSM": "TSM",
    "VIX": "^VIX",
    "TNX": "^TNX",
    "USDKRW": "KRW=X",
    "DXY": "DX-Y.NYB",
    "GOLD": "GC=F",
    "OIL": "CL=F",
}

EXTERNAL_FEATURE_TYPES = {
    "QQQ": "price",
    "SPY": "price",
    "SOXX": "price",
    "NVDA": "price",
    "TSM": "price",
    "VIX": "risk",
    "TNX": "rate",
    "USDKRW": "price",
    "DXY": "price",
    "GOLD": "price",
    "OIL": "price",
}

model_configs = [
    {"model_name": "random_forest", "model_params": {"n_estimators": 500, "max_depth": None}},
    {"model_name": "extra_trees", "model_params": {"n_estimators": 500, "max_depth": None}},
    {"model_name": "gradient_boosting", "model_params": {"n_estimators": 300, "learning_rate": 0.03, "max_depth": 3}},
    {"model_name": "hist_gradient_boosting", "model_params": {"max_iter": 300, "learning_rate": 0.03}},
    {"model_name": "logistic", "model_params": {"C": 1.0}},
    {"model_name": "svc", "model_params": {"C": 1.0, "kernel": "rbf"}},
]


from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline


In [2]:
import requests

verify_ssl=False
session = requests.Session()
session.verify = verify_ssl
session.headers.update({
    "User-Agent": "Mozilla/5.0"
})

original_get = requests.get

def custom_get(*args, **kwargs):
    kwargs["verify"] = verify_ssl
    return session.get(*args, **kwargs)

# FinanceDataReader 내부 requests.get 일시 덮어쓰기
requests.get = custom_get

In [3]:
## 함수

################################################################################################################################################
## 1. Base Feature Dataset 생성
################################################################################################################################################
def load_price_data(ticker, start_date="2020-01-01", end_date=None):
    """
    FinanceDataReader로 가격 데이터를 가져온다.
    Date 컬럼을 일반 컬럼으로 유지한다.
    """
    df = fdr.DataReader(ticker, start_date, end_date)
    df = df.reset_index().rename(columns={"index": "Date"})

    # 컬럼명 정리
    df["Date"] = pd.to_datetime(df["Date"])

    return df

def make_target_etf_features(etf_df, prefix):
    """
    예측 대상 ETF용 feature 생성.
    target은 여기서 만들지 않는다.
    """
    df = etf_df.copy()
    df = df.sort_values("Date").reset_index(drop=True)

    close = df["Adj Close"]
    volume = df["Volume"]

    result = pd.DataFrame()
    result["Date"] = df["Date"]

    # target 만들 때 필요하므로 Close는 반드시 보존
    result[f"{prefix}_adj_close"] = close

    # 수익률
    result[f"{prefix}_ret_1d"] = close.pct_change(1)
    result[f"{prefix}_ret_5d"] = close.pct_change(5)
    result[f"{prefix}_ret_20d"] = close.pct_change(20)

    # 이동평균 대비 위치
    ma_5 = close.rolling(5).mean()
    ma_20 = close.rolling(20).mean()
    ma_60 = close.rolling(60).mean()

    result[f"{prefix}_ma5_ratio"] = close / ma_5 - 1
    result[f"{prefix}_ma20_ratio"] = close / ma_20 - 1
    result[f"{prefix}_ma60_ratio"] = close / ma_60 - 1

    # 변동성
    result[f"{prefix}_vol_20d"] = result[f"{prefix}_ret_1d"].rolling(20).std()

    # 거래량 비율
    vol_ma20 = volume.rolling(20).mean()
    result[f"{prefix}_volume_ratio_20d"] = volume / vol_ma20 - 1

    return result

def make_external_features(raw_df, name, feature_type="price"):
    """
    외부 지표용 최소 파생변수 생성.
    feature_type:
        - price: 일반 가격형 지표
        - risk: VIX 같은 리스크 레벨 지표
        - rate: 금리 지표
    """
    df = raw_df.copy()
    df = df.sort_values("Date").reset_index(drop=True)

    close = df["Adj Close"]

    result = pd.DataFrame()
    result["Date"] = df["Date"]

    if feature_type == "price":
        result[f"{name}_ret_5d"] = close.pct_change(5)
        result[f"{name}_ret_20d"] = close.pct_change(20)

    elif feature_type == "risk":
        result[f"{name}_level"] = close
        result[f"{name}_chg_5d"] = close.diff(5)
        result[f"{name}_chg_20d"] = close.diff(20)

    elif feature_type == "rate":
        result[f"{name}_level"] = close
        result[f"{name}_diff_5d"] = close.diff(5)
        result[f"{name}_diff_20d"] = close.diff(20)

    else:
        raise ValueError("feature_type must be one of ['price', 'risk', 'rate']")

    return result

def make_base_feature_dataset(
    etf_code,
    external_tickers,
    external_feature_types,
    start_date="2020-01-01",
    end_date=None
):
    """
    target 없는 기본 feature dataset 생성.
    이 함수는 느린 작업이므로 한 번만 실행하는 것을 목표로 한다.
    """
    # 1. ETF 본체 로드
    etf_raw = load_price_data(etf_code, start_date, end_date)

    # 2. ETF 본체 feature 생성
    base_df = make_target_etf_features(etf_raw, prefix=etf_code)

    # 3. 외부 지표 붙이기
    for name, ticker in external_tickers.items():
        print(f"Loading external ticker: {name} / {ticker}")

        try:
            raw = load_price_data(ticker, start_date, end_date)

            feature_type = external_feature_types.get(name, "price")

            ext_feat = make_external_features(
                raw_df=raw,
                name=name,
                feature_type=feature_type
            )

            base_df = base_df.merge(ext_feat, on="Date", how="left")

        except Exception as e:
            print(f"[SKIP] {name} / {ticker} 로드 실패:", e)

    # 4. 날짜 정렬
    base_df = base_df.sort_values("Date").reset_index(drop=True)

    # 5. feature_cols 정리
    close_col = f"{etf_code}_adj_close"

    feature_cols = [
        col for col in base_df.columns
        if col not in ["Date", close_col]
    ]

    return base_df, feature_cols, close_col

def make_base_feature_dataset(
    etf_code,
    external_tickers,
    external_feature_types,
    start_date="2020-01-01",
    end_date=None
):
    """
    target 없는 기본 feature dataset 생성.
    이 함수는 느린 작업이므로 한 번만 실행하는 것을 목표로 한다.

    주말 데이터가 섞여 NA가 늘어나는 문제를 막기 위해
    ETF / 외부 ticker 모두 영업일(월~금)만 사용한다.
    """

    # =========================================================
    # 0. 영업일 필터 함수
    # =========================================================
    def keep_weekdays_only(df, date_col="Date"):
        df = df.copy()
        df[date_col] = pd.to_datetime(df[date_col])
        df = df[df[date_col].dt.weekday < 5]  # 월=0, 금=4
        df = df.sort_values(date_col).reset_index(drop=True)
        return df

    # =========================================================
    # 1. ETF 본체 로드
    # =========================================================
    etf_raw = load_price_data(etf_code, start_date, end_date)

    # 주말 제거
    etf_raw = keep_weekdays_only(etf_raw, date_col="Date")

    # =========================================================
    # 2. ETF 본체 feature 생성
    # =========================================================
    base_df = make_target_etf_features(etf_raw, prefix=etf_code)

    # feature 생성 후에도 혹시 모르니 다시 주말 제거
    base_df = keep_weekdays_only(base_df, date_col="Date")

    # =========================================================
    # 3. 외부 지표 붙이기
    # =========================================================
    for name, ticker in external_tickers.items():
        print(f"Loading external ticker: {name} / {ticker}")

        try:
            raw = load_price_data(ticker, start_date, end_date)

            # 외부 ticker도 주말 제거
            raw = keep_weekdays_only(raw, date_col="Date")

            feature_type = external_feature_types.get(name, "price")

            ext_feat = make_external_features(
                raw_df=raw,
                name=name,
                feature_type=feature_type
            )

            # 외부 feature 생성 후에도 다시 주말 제거
            ext_feat = keep_weekdays_only(ext_feat, date_col="Date")

            # ETF 거래일 기준으로 붙임
            base_df = base_df.merge(ext_feat, on="Date", how="left")

        except Exception as e:
            print(f"[SKIP] {name} / {ticker} 로드 실패:", e)

    # =========================================================
    # 4. 날짜 정렬 + 주말 최종 제거
    # =========================================================
    base_df = keep_weekdays_only(base_df, date_col="Date")
    
    # =========================================================
    # 4.1. 외부 지표 NA 보정
    # - ETF 본체는 건드리지 않고
    # - 외부 지표만 ffill
    # =========================================================
    etf_prefix = f"{etf_code}_"

    external_cols = [
        col for col in base_df.columns
        if col != "Date" and not col.startswith(etf_prefix)
    ]

    base_df[external_cols] = base_df[external_cols].ffill()

    # =========================================================
    # 5. feature_cols 정리
    # =========================================================
    close_col = f"{etf_code}_adj_close"

    feature_cols = [
        col for col in base_df.columns
        if col not in ["Date", close_col]
    ]

    return base_df, feature_cols, close_col

################################################################################################################################################
## 2. VIF 기반 불필요 칼럼 제거
################################################################################################################################################
def reduce_features_by_vif(
    df,
    feature_cols,
    vif_threshold=30.0,
    date_col="Date",
    verbose=True
):
    """
    VIF 기준으로 다중공선성이 높은 feature를 반복 제거한다.

    주의:
    - target 생성 전 단계에서 실행한다.
    - lag 생성 전 단계에서 실행한다.
    - Date, adj_close 등 보존 컬럼은 feature_cols에 넣지 않는 것을 전제로 한다.
    """

    # 1. 숫자형 feature만 사용
    numeric_feature_cols = [
        col for col in feature_cols
        if col in df.columns and pd.api.types.is_numeric_dtype(df[col])
    ]

    work_df = df[numeric_feature_cols].copy()

    # 2. inf 처리
    work_df = work_df.replace([np.inf, -np.inf], np.nan)

    # 3. VIF 계산용 결측 제거
    #    여기서는 VIF 계산에만 dropna를 쓰고,
    #    원본 df 자체를 줄이지는 않는다.
    vif_calc_df = work_df.dropna(axis=0).copy()

    print("VIF 계산 대상 row 수:", len(vif_calc_df))
    print("VIF 계산 대상 feature 수:", len(numeric_feature_cols))

    if len(vif_calc_df) == 0:
        raise ValueError("VIF 계산 가능한 데이터가 없습니다. 결측값을 확인하세요.")

    # 4. 상수 컬럼 제거
    nunique = vif_calc_df.nunique()
    constant_cols = nunique[nunique <= 1].index.tolist()

    if len(constant_cols) > 0:
        print("상수 컬럼 제거:", constant_cols)

    remaining_cols = [
        col for col in numeric_feature_cols
        if col not in constant_cols
    ]

    removed_records = []

    # 5. VIF 반복 제거
    while True:
        if len(remaining_cols) <= 1:
            break

        X = vif_calc_df[remaining_cols].copy()

        # 표준화
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        vif_values = []

        for i, col in enumerate(remaining_cols):
            try:
                vif = variance_inflation_factor(X_scaled, i)
            except Exception:
                vif = np.inf

            vif_values.append({
                "feature": col,
                "vif": vif
            })

        vif_df = pd.DataFrame(vif_values).sort_values("vif", ascending=False)

        max_vif_row = vif_df.iloc[0]
        max_feature = max_vif_row["feature"]
        max_vif = max_vif_row["vif"]

        if verbose:
            print(f"현재 max VIF: {max_vif:.2f} / feature: {max_feature}")

        if max_vif <= vif_threshold:
            break

        # 가장 VIF 높은 컬럼 제거
        remaining_cols.remove(max_feature)

        removed_records.append({
            "removed_feature": max_feature,
            "vif": max_vif,
            "remaining_feature_count": len(remaining_cols)
        })

    removed_vif_df = pd.DataFrame(removed_records)

    # 6. 최종 VIF 테이블 계산
    final_vif_records = []

    if len(remaining_cols) > 1:
        X_final = vif_calc_df[remaining_cols].copy()

        scaler = StandardScaler()
        X_final_scaled = scaler.fit_transform(X_final)

        for i, col in enumerate(remaining_cols):
            try:
                vif = variance_inflation_factor(X_final_scaled, i)
            except Exception:
                vif = np.inf

            final_vif_records.append({
                "feature": col,
                "vif": vif
            })

        final_vif_df = pd.DataFrame(final_vif_records).sort_values("vif", ascending=False)

    else:
        final_vif_df = pd.DataFrame({
            "feature": remaining_cols,
            "vif": [np.nan] * len(remaining_cols)
        })

    print()
    print("========== VIF 제거 결과 ==========")
    print("초기 feature 수:", len(numeric_feature_cols))
    print("상수 제거 feature 수:", len(constant_cols))
    print("VIF 제거 feature 수:", len(removed_records))
    print("최종 feature 수:", len(remaining_cols))
    print("===================================")

    return remaining_cols, removed_vif_df, final_vif_df


################################################################################################################################################
## 3. Target 변수 생성
################################################################################################################################################
def add_target_column(
    df,
    close_col,
    n_days=5,
    threshold=0.05,
    target_col=None
):
    """
    현재 시점 기준 n_days 뒤 수익률이 threshold 이상이면 1, 아니면 0인 target 생성.

    예:
    n_days=5, threshold=0.05
    → 5거래일 뒤 수익률이 +5% 이상이면 target=1
    """

    result = df.copy()
    result = result.sort_values("Date").reset_index(drop=True)

    if target_col is None:
        target_col = f"target_{n_days}d_up_{int(threshold * 100)}pct"

    # 미래 가격
    future_close = result[close_col].shift(-n_days)

    # 미래 수익률
    result[f"future_ret_{n_days}d"] = future_close / result[close_col] - 1

    # target 생성
    result[target_col] = np.where(
        result[f"future_ret_{n_days}d"] >= threshold,
        1,
        0
    )

    # 마지막 n_days개는 미래 가격이 없으므로 제거
    result.loc[result[f"future_ret_{n_days}d"].isna(), target_col] = np.nan

    return result, target_col


################################################################################################################################################
## 4. Lag 생성 후 변수별 최적 lag 탐색
################################################################################################################################################

def find_best_lag_by_feature(
    df,
    feature_cols,
    target_col,
    lag_days=[1, 3, 5, 10, 20],
    date_col="Date",
    method="corr"
):
    """
    각 feature별로 target과 가장 관계가 강한 선행 lag를 찾는다.

    lag 의미:
    - lag=1  : feature의 1거래일 전 값으로 오늘 target 설명
    - lag=5  : feature의 5거래일 전 값으로 오늘 target 설명
    - lag=20 : feature의 20거래일 전 값으로 오늘 target 설명

    즉, feature가 먼저 움직이고 나중에 target이 움직이는 구조만 본다.
    """

    records = []

    for col in feature_cols:
        if col not in df.columns:
            continue

        for lag in lag_days:
            temp = df[[date_col, col, target_col]].copy()

            # 선행변수 구조
            temp[f"{col}_lag{lag}"] = temp[col].shift(lag)

            temp = temp[[f"{col}_lag{lag}", target_col]].replace(
                [np.inf, -np.inf],
                np.nan
            ).dropna()

            if len(temp) < 30:
                continue

            x = temp[f"{col}_lag{lag}"]
            y = temp[target_col]

            if x.nunique() <= 1:
                corr = np.nan
            else:
                corr = x.corr(y)

            records.append({
                "feature": col,
                "lag": lag,
                "corr": corr,
                "abs_corr": abs(corr) if pd.notna(corr) else np.nan,
                "n_rows": len(temp)
            })

    lag_result_df = pd.DataFrame(records)

    if lag_result_df.empty:
        raise ValueError("lag 탐색 결과가 비어 있습니다. feature_cols 또는 target_col을 확인하세요.")

    # feature별 abs_corr가 가장 큰 lag 선택
    best_lag_df = (
        lag_result_df
        .sort_values(["feature", "abs_corr"], ascending=[True, False])
        .groupby("feature", as_index=False)
        .head(1)
        .reset_index(drop=True)
    )

    best_lag_df = best_lag_df.sort_values("abs_corr", ascending=False).reset_index(drop=True)

    return lag_result_df, best_lag_df

def make_lagged_dataset_by_best_lag(
    df,
    best_lag_df,
    target_col,
    close_col,
    n_days=5,
    date_col="Date"
):
    """
    best_lag_df 기준으로 feature별 최적 lag를 적용한 최종 모델용 데이터셋 생성.
    """

    result = pd.DataFrame()
    result[date_col] = df[date_col]
    result[close_col] = df[close_col]

    # 확인용 미래수익률 보존
    future_ret_col = f"future_ret_{n_days}d"
    if future_ret_col in df.columns:
        result[future_ret_col] = df[future_ret_col]

    # target 보존
    result[target_col] = df[target_col]

    lagged_feature_cols = []

    for _, row in best_lag_df.iterrows():
        feature = row["feature"]
        lag = int(row["lag"])

        if feature not in df.columns:
            continue

        lagged_col = f"{feature}_lag{lag}"
        result[lagged_col] = df[feature].shift(lag)
        lagged_feature_cols.append(lagged_col)

    # 결측/무한값 제거
    result = result.replace([np.inf, -np.inf], np.nan)
    result = result.dropna().reset_index(drop=True)

    return result, lagged_feature_cols



################################################################################################################################################
## 5. RandomForest in-sample 학습 + permutation importance
################################################################################################################################################

def run_rf_permutation_importance_in_sample(
    lagged_df,
    feature_cols,
    target_col,
    date_col="Date",
    close_col=None,
    n_rf_runs=3,
    n_repeats=10,
    random_state=42
):
    """
    lagged_df 기준으로 RandomForestClassifier를 in-sample 학습한 뒤
    permutation importance를 반복 계산한다.

    핵심:
    - RF를 n_rf_runs번 학습
    - 각 RF마다 permutation을 n_repeats번 수행
    - feature별 importance raw 값을 전부 저장
    - 최종 importance_df는 총 n_rf_runs * n_repeats개의 raw importance 기준으로 계산

    예:
    n_rf_runs=3, n_repeats=10이면
    feature별 importance 값 30개를 기반으로 평균/표준편차/스코어 계산
    """

    df = lagged_df.copy()

    # =====================================================
    # 1. 모델 input / target 분리
    # =====================================================

    X = df[feature_cols].copy()
    y = df[target_col].copy()

    X = X.replace([np.inf, -np.inf], np.nan)

    model_df = pd.concat([X, y], axis=1).dropna().copy()

    X = model_df[feature_cols].copy()
    y = model_df[target_col].astype(int).copy()


    # =====================================================
    # 2. 여러 RF run + permutation raw importance 저장
    # =====================================================

    importance_records = []
    baseline_records = []

    for run in range(n_rf_runs):
        print()
        print(f"========== RF RUN {run + 1} / {n_rf_runs} ==========")

        rf = RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features="sqrt",
            class_weight="balanced",
            random_state=random_state + run,
            n_jobs=-1
        )

        rf.fit(X, y)

        # =================================================
        # 3. in-sample 예측 성능 확인
        # =================================================

        pred = rf.predict(X)
        pred_proba = rf.predict_proba(X)[:, 1]

        acc = accuracy_score(y, pred)
        precision = precision_score(y, pred, zero_division=0)
        recall = recall_score(y, pred, zero_division=0)
        f1 = f1_score(y, pred, zero_division=0)
        loss = log_loss(y, pred_proba)

        cm = confusion_matrix(y, pred)

        print("accuracy :", round(acc, 4))
        print("precision:", round(precision, 4))
        print("recall   :", round(recall, 4))
        print("f1       :", round(f1, 4))
        print("log_loss :", round(loss, 4))

        baseline_records.append({
            "run": run + 1,
            "accuracy": acc,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "log_loss": loss,
            "pred_1_count": int((pred == 1).sum()),
            "actual_1_count": int((y == 1).sum())
        })

        # =================================================
        # 4. permutation importance
        # =================================================

        perm = permutation_importance(
            rf,
            X,
            y,
            scoring="neg_log_loss",
            n_repeats=n_repeats,
            random_state=random_state + run,
            n_jobs=-1
        )

        # 핵심:
        # perm.importances shape = (n_features, n_repeats)
        # 여기서 반복별 raw importance를 전부 저장한다.
        for i, col in enumerate(feature_cols):
            for repeat_idx, importance_value in enumerate(perm.importances[i]):
                importance_records.append({
                    "run": run + 1,
                    "repeat": repeat_idx + 1,
                    "feature": col,
                    "importance": importance_value
                })

    # =====================================================
    # 5. 결과 정리
    # =====================================================

    raw_importance_df = pd.DataFrame(importance_records)
    baseline_df = pd.DataFrame(baseline_records)

    importance_df = (
        raw_importance_df
        .groupby("feature", as_index=False)
        .agg(
            importance_mean=("importance", "mean"),
            importance_std=("importance", "std"),
            importance_var=("importance", "var"),
            importance_min=("importance", "min"),
            importance_max=("importance", "max"),
            run_count=("run", "nunique"),
            repeat_count=("repeat", "count")
        )
    )

    # 안정성 점수
    # 평균 중요도는 높고, 30회 전체 기준 표준편차는 낮을수록 높게
    importance_df["importance_score"] = (
        importance_df["importance_mean"]
        - importance_df["importance_std"].fillna(0)
    )

    importance_df = importance_df.sort_values(
        "importance_score",
        ascending=False
    ).reset_index(drop=True)

    return importance_df, raw_importance_df, baseline_df

def split_lagged_feature_name(feature_name):
    """
    feature_lag20 형태의 컬럼명을 원본 feature와 lag로 분리한다.
    """

    if "_lag" not in feature_name:
        return feature_name, np.nan

    base_name = feature_name.rsplit("_lag", 1)[0]
    lag = feature_name.rsplit("_lag", 1)[1]

    try:
        lag = int(lag)
    except:
        lag = np.nan

    return base_name, lag



In [4]:

# =========================================================
# 실행부 함수화
# - 1) top_feature_df 생성 함수
# - 2) valid 검증/예측 함수
# =========================================================

def build_top_feature_df(
    etf_code,
    n_days,
    threshold,
    lag_search_years,
    random_state=42,
    n_rf_runs=3,
    n_repeats=10,
    top_n=30,
    start_date=START_DATE,
    end_date=END_DATE,
    valid_months=VALID_MONTHS,
    external_tickers=EXTERNAL_TICKERS,
    external_feature_types=EXTERNAL_FEATURE_TYPES,
    vif_threshold=VIF_THRESHOLD,
    lag_days=LAG_DAYS,
    verbose=True
):
    """
    base_df 생성 → train/valid 분리 → VIF 제거 → target 생성
    → lag 탐색 → lagged_df 생성 → RF permutation importance
    → top_feature_df 추출까지 한 번에 수행한다.

    Returns
    -------
    result : dict
        검증 함수에서 다시 필요한 객체들을 모두 담아서 반환한다.
        주요 key:
        - top_feature_df
        - base_df
        - train_base_df
        - valid_base_df
        - close_col
        - target_col
        - lagged_df
        - lagged_feature_cols
        - best_lag_df
        - importance_df
        - importance_with_lag_df
    """

    # =========================================================
    # 1. target 없는 base dataset 생성
    # =========================================================
    if verbose:
        print()
        print("=" * 50)
        print("1. Base feature dataset created.")

    base_df, base_feature_cols, close_col = make_base_feature_dataset(
        etf_code=etf_code,
        external_tickers=external_tickers,
        external_feature_types=external_feature_types,
        start_date=start_date,
        end_date=end_date
    )

    max_date = base_df["Date"].max()
    valid_start_date = max_date - pd.DateOffset(months=valid_months)

    train_base_df = base_df[base_df["Date"] < valid_start_date].copy()
    valid_base_df = base_df[base_df["Date"] >= valid_start_date].copy()

    if verbose:
        print("train_base_df shape:", train_base_df.shape)
        print("valid_base_df shape:", valid_base_df.shape)
        print("close_col:", close_col)
        print("=" * 50)

    # =========================================================
    # 2. VIF 기반 불필요 칼럼 제거
    # =========================================================
    if verbose:
        print()
        print("=" * 50)
        print("2. VIF filtering completed.")

    vif_feature_cols, removed_vif_df, final_vif_df = reduce_features_by_vif(
        df=train_base_df,
        feature_cols=base_feature_cols,
        vif_threshold=vif_threshold,
        date_col="Date",
        verbose=verbose
    )

    keep_cols = ["Date", close_col] + vif_feature_cols
    vif_filtered_df = train_base_df[keep_cols].copy()

    if verbose:
        print("vif_filtered_df shape:", vif_filtered_df.shape)
        print("제거된 컬럼:")
        print(removed_vif_df)
        print("=" * 50)

    # =========================================================
    # 3. Target 변수 생성
    # =========================================================
    if verbose:
        print()
        print("=" * 50)
        print("3. Target column added.")

    target_df, target_col = add_target_column(
        df=vif_filtered_df,
        close_col=close_col,
        n_days=n_days,
        threshold=threshold
    )

    # target 없는 마지막 n_days 행 제거
    target_df = target_df.dropna(subset=[target_col]).copy()
    target_df[target_col] = target_df[target_col].astype(int)

    if verbose:
        print("target_col:", target_col)
        print("target_df shape after dropna:", target_df.shape)
        print("target 분포:")
        print(target_df[target_col].value_counts())
        print("target 비율:")
        print(target_df[target_col].value_counts(normalize=True))
        print("=" * 50)

    # =========================================================
    # 4. 최근 lag_search_years 기준으로 lag 탐색
    # =========================================================
    if verbose:
        print()
        print("=" * 50)
        print("4. 변수별 최적 LAG 탐색 완료.")

    max_date = target_df["Date"].max()
    lag_search_start_date = max_date - pd.DateOffset(years=lag_search_years)

    target_df_for_lag_search = target_df[
        target_df["Date"] >= lag_search_start_date
    ].copy()

    if verbose:
        print("lag 탐색 기준 기간:")
        print(target_df_for_lag_search["Date"].min(), "~", target_df_for_lag_search["Date"].max())
        print("lag 탐색용 데이터 shape:", target_df_for_lag_search.shape)

    exclude_cols_for_lag = [
        "Date",
        close_col,
        f"future_ret_{n_days}d",
        target_col
    ]

    lag_search_feature_cols = [
        col for col in target_df.columns
        if col not in exclude_cols_for_lag
    ]

    lag_result_df, best_lag_df = find_best_lag_by_feature(
        df=target_df_for_lag_search,
        feature_cols=lag_search_feature_cols,
        target_col=target_col,
        lag_days=lag_days,
        date_col="Date"
    )

    if verbose:
        print("전체 lag 탐색 결과 shape:", lag_result_df.shape)

    # =========================================================
    # 5. best lag 적용해서 lagged_df 생성
    # =========================================================
    lagged_df, lagged_feature_cols = make_lagged_dataset_by_best_lag(
        df=target_df,
        best_lag_df=best_lag_df,
        target_col=target_col,
        close_col=close_col,
        n_days=n_days,
        date_col="Date"
    )

    if verbose:
        print("lagged_df shape:", lagged_df.shape)
        print("lagged feature count:", len(lagged_feature_cols))
        print("=" * 50)

    # =========================================================
    # 6. permutation importance 실행
    # =========================================================
    if verbose:
        print()
        print("=" * 50)
        print("5. RandomForest in-sample 학습 + permutation importance 완료.")

    importance_df, raw_importance_df, baseline_df = run_rf_permutation_importance_in_sample(
        lagged_df=lagged_df,
        feature_cols=lagged_feature_cols,
        target_col=target_col,
        date_col="Date",
        close_col=close_col,
        n_rf_runs=n_rf_runs,
        n_repeats=n_repeats,
        random_state=random_state
    )

    # =========================================================
    # 7. 결과 feature명 / lag 분리
    # =========================================================
    importance_view_df = importance_df.copy()

    importance_view_df[["base_feature", "selected_lag"]] = importance_view_df["feature"].apply(
        lambda x: pd.Series(split_lagged_feature_name(x))
    )

    importance_view_df = importance_view_df[
        [
            "feature",
            "base_feature",
            "selected_lag",
            "importance_score",
            "importance_mean",
            "importance_std",
            "importance_var",
            "importance_min",
            "importance_max",
            "run_count",
            "repeat_count"
        ]
    ]

    # =========================================================
    # 8. best_lag_df와 importance 결과 합치기
    # =========================================================
    importance_with_lag_df = importance_view_df.merge(
        best_lag_df.rename(columns={
            "feature": "base_feature",
            "lag": "best_lag",
            "corr": "lag_corr",
            "abs_corr": "lag_abs_corr",
            "n_rows": "lag_n_rows"
        }),
        on="base_feature",
        how="left"
    )

    importance_with_lag_df = importance_with_lag_df[
        [
            "feature",
            "base_feature",
            "selected_lag",
            "importance_score",
            "importance_mean",
            "importance_std",
            "importance_var",
            "importance_min",
            "importance_max",
            "best_lag",
            "lag_corr",
            "lag_abs_corr",
            "lag_n_rows",
            "run_count",
            "repeat_count"
        ]
    ]

    importance_with_lag_df = importance_with_lag_df.sort_values(
        "importance_score",
        ascending=False
    ).reset_index(drop=True)

    # =========================================================
    # 9. TOP 변수 추출
    # =========================================================
    top_feature_df = importance_with_lag_df.head(top_n).copy()
    top_feature_cols = top_feature_df["feature"].tolist()

    if verbose:
        print("TOP feature count:", len(top_feature_cols))
        print(top_feature_cols)
        print("=" * 50)
        display(top_feature_df)

    result = {
        "etf_code": etf_code,
        "n_days": n_days,
        "threshold": threshold,
        "lag_search_years": lag_search_years,
        "random_state": random_state,
        "n_rf_runs": n_rf_runs,
        "n_repeats": n_repeats,
        "top_n": top_n,
        "start_date": start_date,
        "end_date": end_date,
        "valid_months": valid_months,
        "vif_threshold": vif_threshold,
        "lag_days": lag_days,

        "base_df": base_df,
        "base_feature_cols": base_feature_cols,
        "train_base_df": train_base_df,
        "valid_base_df": valid_base_df,
        "close_col": close_col,

        "vif_feature_cols": vif_feature_cols,
        "removed_vif_df": removed_vif_df,
        "final_vif_df": final_vif_df,
        "vif_filtered_df": vif_filtered_df,

        "target_df": target_df,
        "target_col": target_col,

        "lag_result_df": lag_result_df,
        "best_lag_df": best_lag_df,
        "lagged_df": lagged_df,
        "lagged_feature_cols": lagged_feature_cols,

        "importance_df": importance_df,
        "raw_importance_df": raw_importance_df,
        "baseline_df": baseline_df,
        "importance_with_lag_df": importance_with_lag_df,

        "top_feature_df": top_feature_df,
        "top_feature_cols": top_feature_cols
    }

    return result



# =========================================================
# 검증용 분류 모델 생성 함수
# - top_feature_df를 뽑는 모델은 RF로 유지
# - valid 검증 단계에서만 model_name으로 모델을 바꿔 테스트
# =========================================================

def make_classifier_model(model_name="random_forest", random_state=42, model_params=None):
    """
    검증/예측 단계에서 사용할 분류모델을 생성한다.

    Parameters
    ----------
    model_name : str
        사용할 모델 이름.
        지원 모델:
        - "random_forest"
        - "extra_trees"
        - "gradient_boosting"
        - "hist_gradient_boosting"
        - "logistic"
        - "svc"
        - "knn"
    random_state : int
        random_state를 지원하는 모델에 적용.
    model_params : dict or None
        모델별 파라미터 덮어쓰기용 dict.

    Returns
    -------
    model : sklearn estimator
        fit / predict_proba 가능한 분류모델.
    """

    if model_params is None:
        model_params = {}

    model_name = model_name.lower()

    if model_name == "random_forest":
        default_params = dict(
            n_estimators=500,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features="sqrt",
            class_weight="balanced",
            random_state=random_state,
            n_jobs=-1
        )
        default_params.update(model_params)
        model = RandomForestClassifier(**default_params)

    elif model_name == "extra_trees":
        default_params = dict(
            n_estimators=500,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features="sqrt",
            class_weight="balanced",
            random_state=random_state,
            n_jobs=-1
        )
        default_params.update(model_params)
        model = ExtraTreesClassifier(**default_params)

    elif model_name == "gradient_boosting":
        default_params = dict(
            n_estimators=300,
            learning_rate=0.03,
            max_depth=3,
            random_state=random_state
        )
        default_params.update(model_params)
        model = GradientBoostingClassifier(**default_params)

    elif model_name == "hist_gradient_boosting":
        default_params = dict(
            max_iter=300,
            learning_rate=0.03,
            max_leaf_nodes=31,
            random_state=random_state
        )
        default_params.update(model_params)
        model = HistGradientBoostingClassifier(**default_params)

    elif model_name == "logistic":
        default_params = dict(
            C=1.0,
            class_weight="balanced",
            max_iter=3000,
            random_state=random_state
        )
        default_params.update(model_params)
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(**default_params)
        )

    elif model_name == "svc":
        default_params = dict(
            C=1.0,
            kernel="rbf",
            gamma="scale",
            class_weight="balanced",
            probability=True,
            random_state=random_state
        )
        default_params.update(model_params)
        model = make_pipeline(
            StandardScaler(),
            SVC(**default_params)
        )

    elif model_name == "knn":
        default_params = dict(
            n_neighbors=15,
            weights="distance"
        )
        default_params.update(model_params)
        model = make_pipeline(
            StandardScaler(),
            KNeighborsClassifier(**default_params)
        )

    else:
        raise ValueError(
            "지원하지 않는 model_name입니다. "
            "사용 가능: random_forest, extra_trees, gradient_boosting, "
            "hist_gradient_boosting, logistic, svc, knn"
        )

    return model


def safe_binary_metrics(y_true, pred, pred_proba=None):
    """
    valid 구간에 한 클래스만 있는 경우에도 에러 없이 metric을 계산한다.
    """
    y_true = pd.Series(y_true).astype(int)
    pred = pd.Series(pred).astype(int)

    accuracy = accuracy_score(y_true, pred)
    precision = precision_score(y_true, pred, zero_division=0)
    recall = recall_score(y_true, pred, zero_division=0)
    f1 = f1_score(y_true, pred, zero_division=0)

    auc = np.nan
    valid_logloss = np.nan

    if pred_proba is not None:
        pred_proba = np.asarray(pred_proba)
        if y_true.nunique() == 2:
            auc = roc_auc_score(y_true, pred_proba)
            valid_logloss = log_loss(y_true, pred_proba, labels=[0, 1])

    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc,
        "logloss": valid_logloss,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }


def validate_top_feature_df(
    feature_result,
    model_name="random_forest",
    model_params=None,
    pred_threshold=0.5,
    random_state=42,
    n_days=None,
    threshold=None,
    verbose=True
):
    """
    build_top_feature_df() 결과를 받아서 valid_df 생성 후
    Train 전체 학습 → valid_df 예측 → precision 확인까지 수행한다.

    Parameters
    ----------
    feature_result : dict
        build_top_feature_df()에서 반환한 dict.
    model_name : str
        검증에 사용할 분류모델 이름.
    model_params : dict or None
        모델별 파라미터 덮어쓰기용 dict.
    pred_threshold : float
        예측 확률을 1로 바꿀 기준값.
    random_state : int
        RandomForest random_state.
    n_days : int or None
        None이면 feature_result의 n_days 사용.
    threshold : float or None
        None이면 feature_result의 threshold 사용.

    Returns
    -------
    result : dict
        - valid_df
        - valid_pred_df
        - eval_df
        - pred_1_df
        - metric_df
        - model
        - model_name
        - model_params
    """

    # =========================================================
    # 0. 필요한 객체 꺼내기
    # =========================================================
    top_feature_df = feature_result["top_feature_df"].copy()
    base_df = feature_result["base_df"].copy()
    valid_base_df = feature_result["valid_base_df"].copy()
    lagged_df = feature_result["lagged_df"].copy()
    close_col = feature_result["close_col"]
    target_col = feature_result["target_col"]

    if n_days is None:
        n_days = feature_result["n_days"]

    if threshold is None:
        threshold = feature_result["threshold"]

    # =========================================================
    # 1. valid_df 생성
    # =========================================================
    if verbose:
        print("=" * 60)
        print("6. valid_df 생성")
        print("=" * 60)

    if ("base_feature" not in top_feature_df.columns) or ("selected_lag" not in top_feature_df.columns):
        top_feature_df[["base_feature", "selected_lag"]] = top_feature_df["feature"].apply(
            lambda x: pd.Series(split_lagged_feature_name(x))
        )

    top_feature_df["selected_lag"] = top_feature_df["selected_lag"].astype(int)

    top_feature_cols = top_feature_df["feature"].tolist()
    top_base_features = top_feature_df["base_feature"].unique().tolist()

    if verbose:
        print("top feature 수:", len(top_feature_cols))

    need_cols = ["Date", close_col] + top_base_features
    missing_cols = [col for col in need_cols if col not in base_df.columns]

    if len(missing_cols) > 0:
        raise ValueError(f"base_df에 없는 컬럼이 있습니다: {missing_cols}")

    valid_df = base_df[need_cols].copy()
    valid_df = valid_df.sort_values("Date").reset_index(drop=True)

    # 전체 base_df 기준으로 lag 생성해야 최근 valid 구간의 lag가 계산됨
    for _, row in top_feature_df.iterrows():
        base_feature = row["base_feature"]
        selected_lag = int(row["selected_lag"])
        lagged_feature = row["feature"]

        valid_df[lagged_feature] = valid_df[base_feature].shift(selected_lag)

    if verbose:
        print("lag 생성 후 valid_df shape:", valid_df.shape)

    valid_df = valid_df[["Date", close_col] + top_feature_cols].copy()

    if verbose:
        print("원본 base_feature 제거 후 valid_df shape:", valid_df.shape)
        print("최종 컬럼 수:", len(valid_df.columns))

    valid_df, _ = add_target_column(
        df=valid_df,
        close_col=close_col,
        n_days=n_days,
        threshold=threshold,
        target_col=target_col
    )

    if verbose:
        print("target 생성 후 valid_df shape:", valid_df.shape)

    valid_df = valid_df.tail(len(valid_base_df)).copy()
    valid_df = valid_df.reset_index(drop=True)

    if verbose:
        print("최종 valid_df shape:", valid_df.shape)
        print("valid_base_df shape:", valid_base_df.shape)
        print("valid_df 기간:")
        print(valid_df["Date"].min(), "~", valid_df["Date"].max())

    # =========================================================
    # 2. Train 학습 후 valid_df 예측
    # =========================================================
    if verbose:
        print("=" * 60)
        print("7. Train 학습 후 valid_df 예측")
        print("=" * 60)

    missing_train_cols = [col for col in top_feature_cols if col not in lagged_df.columns]
    missing_valid_cols = [col for col in top_feature_cols if col not in valid_df.columns]

    if len(missing_train_cols) > 0:
        raise ValueError(f"lagged_df에 없는 top feature가 있습니다: {missing_train_cols}")

    if len(missing_valid_cols) > 0:
        raise ValueError(f"valid_df에 없는 top feature가 있습니다: {missing_valid_cols}")

    if verbose:
        print("사용 feature 수:", len(top_feature_cols))

    train_df = lagged_df[
        ["Date", close_col, target_col] + top_feature_cols
    ].copy()

    train_df = train_df.replace([np.inf, -np.inf], np.nan)
    train_df = train_df.dropna(subset=top_feature_cols + [target_col]).copy()

    X_train = train_df[top_feature_cols].copy()
    y_train = train_df[target_col].astype(int).copy()

    if verbose:
        print("train_df shape:", train_df.shape)
        print("X_train shape:", X_train.shape)
        print("train target 분포:")
        print(y_train.value_counts())
        print(y_train.value_counts(normalize=True))

    valid_pred_df = valid_df.copy()

    valid_pred_df["pred_proba"] = np.nan
    valid_pred_df["pred"] = np.nan

    valid_available_mask = (
        valid_pred_df[top_feature_cols]
        .replace([np.inf, -np.inf], np.nan)
        .notna()
        .all(axis=1)
    )

    X_valid = valid_pred_df.loc[valid_available_mask, top_feature_cols].copy()

    if verbose:
        print("valid_df shape:", valid_df.shape)
        print("예측 가능한 valid row 수:", len(X_valid))
        print("예측 불가능 row 수:", len(valid_df) - len(X_valid))

    model = make_classifier_model(
        model_name=model_name,
        random_state=random_state,
        model_params=model_params
    )

    model.fit(X_train, y_train)

    if verbose:
        print(f"모델 학습 완료: {model_name}")

    valid_pred_proba = model.predict_proba(X_valid)[:, 1]
    valid_pred = (valid_pred_proba >= pred_threshold).astype(int)

    valid_pred_df.loc[valid_available_mask, "pred_proba"] = valid_pred_proba
    valid_pred_df.loc[valid_available_mask, "pred"] = valid_pred

    valid_pred_df["pred"] = valid_pred_df["pred"].astype("Int64")

    if verbose:
        print("valid_df 예측 완료")

    # =========================================================
    # 3. 예측 결과 확인
    # =========================================================
    display_cols = ["Date", close_col, "pred_proba", "pred"]

    if target_col in valid_pred_df.columns:
        display_cols = ["Date", close_col, target_col, "pred_proba", "pred"]

    if verbose:
        display(valid_pred_df[display_cols])
        print("예측값 분포:")
        print(valid_pred_df["pred"].value_counts(dropna=False))

    # =========================================================
    # 4. pred = 1 기준 precision 확인
    # =========================================================
    if verbose:
        print("=" * 60)
        print("8. 예측 1 기준 정확도 확인")
        print("=" * 60)

    eval_df = valid_pred_df[
        valid_pred_df["pred"].notna() &
        valid_pred_df[target_col].notna()
    ].copy()

    eval_df["pred"] = eval_df["pred"].astype(int)
    eval_df[target_col] = eval_df[target_col].astype(int)

    pred_1_df = eval_df[eval_df["pred"] == 1].copy()

    pred_1_count = len(pred_1_df)
    pred_1_actual_1_count = (pred_1_df[target_col] == 1).sum()

    if pred_1_count > 0:
        pred_1_precision = pred_1_actual_1_count / pred_1_count
    else:
        pred_1_precision = np.nan

    if len(eval_df) > 0:
        metric_dict = safe_binary_metrics(
            y_true=eval_df[target_col],
            pred=eval_df["pred"],
            pred_proba=eval_df["pred_proba"]
        )
    else:
        metric_dict = {
            "accuracy": np.nan,
            "precision": np.nan,
            "recall": np.nan,
            "f1": np.nan,
            "auc": np.nan,
            "logloss": np.nan,
            "tn": np.nan,
            "fp": np.nan,
            "fn": np.nan,
            "tp": np.nan,
        }

    metric_df = pd.DataFrame([{
        "etf_code": feature_result.get("etf_code"),
        "n_days": n_days,
        "threshold": threshold,
        "lag_search_years": feature_result.get("lag_search_years"),
        "top_n": feature_result.get("top_n"),
        "feature_count": len(top_feature_cols),
        "model_name": model_name,
        "model_params": str(model_params),
        "eval_count": len(eval_df),
        "pred_1_count": pred_1_count,
        "pred_1_actual_1_count": pred_1_actual_1_count,
        "pred_1_precision": pred_1_precision,
        "pred_threshold": pred_threshold,
        **metric_dict
    }])

    if verbose:
        print("평가 가능 row 수:", len(eval_df))
        print("예측 1 개수:", pred_1_count)
        print("예측 1 중 실제 1 개수:", pred_1_actual_1_count)
        print("예측 1 기준 정확도 precision:", pred_1_precision)
        display(metric_df)

    result = {
        "valid_df": valid_df,
        "valid_pred_df": valid_pred_df,
        "eval_df": eval_df,
        "pred_1_df": pred_1_df,
        "metric_df": metric_df,
        "model": model,
        "model_name": model_name,
        "model_params": model_params,
        "top_feature_cols": top_feature_cols
    }

    return result



def validate_multiple_models(
    feature_result,
    model_configs=None,
    pred_threshold=0.5,
    random_state=42,
    verbose=True
):
    """
    동일한 top_feature_df / valid_df 조건에서 여러 분류모델을 한 번에 검증한다.

    Parameters
    ----------
    feature_result : dict
        build_top_feature_df() 결과.
    model_configs : list[dict] or None
        예:
        [
            {"model_name": "random_forest", "model_params": {"n_estimators": 300}},
            {"model_name": "extra_trees", "model_params": {"n_estimators": 300}},
            {"model_name": "logistic", "model_params": {"C": 0.5}},
        ]
        None이면 기본 모델 세트를 사용한다.
    pred_threshold : float
        예측확률을 1로 바꿀 기준값.

    Returns
    -------
    result : dict
        - summary_df : 모델별 metric 비교표
        - results : 모델별 상세 결과 dict
    """

    if model_configs is None:
        model_configs = [
            {"model_name": "random_forest", "model_params": None},
            {"model_name": "extra_trees", "model_params": None},
            {"model_name": "gradient_boosting", "model_params": None},
            {"model_name": "hist_gradient_boosting", "model_params": None},
            {"model_name": "logistic", "model_params": None},
            {"model_name": "svc", "model_params": None},
        ]

    results = {}
    metric_list = []

    for cfg in model_configs:
        model_name = cfg.get("model_name")
        model_params = cfg.get("model_params")

        if verbose:
            print("=" * 80)
            print(f"모델 검증 시작: {model_name}")
            print("model_params:", model_params)
            print("=" * 80)

        result = validate_top_feature_df(
            feature_result=feature_result,
            model_name=model_name,
            model_params=model_params,
            pred_threshold=pred_threshold,
            random_state=random_state,
            verbose=verbose
        )

        results[model_name] = result
        metric_list.append(result["metric_df"])

    summary_df = pd.concat(metric_list, ignore_index=True)

    sort_cols = ["pred_1_precision", "precision", "recall", "f1"]
    summary_df = summary_df.sort_values(
        sort_cols,
        ascending=[False, False, False, False]
    ).reset_index(drop=True)

    if verbose:
        print("=" * 80)
        print("모델별 검증 결과 요약")
        print("=" * 80)
        display(summary_df)

    return {
        "summary_df": summary_df,
        "results": results
    }


## 모의투자 시뮬레이션 함수


In [5]:

# =========================================================
# 모의투자 시뮬레이션 함수
# - Excel summary의 0번 row config를 가져와서
# - 사용자가 지정한 기간 동안 하루씩 rolling
# - 각 기준일의 전 거래일까지 학습/feature selection/예측
# - 실제 n_days 뒤 결과와 비교
# =========================================================

import ast
import json
import math
from pathlib import Path
from datetime import datetime

import plotly.graph_objects as go

SIMULATION_RESULT_DIR = Path("simulation_results")
SIMULATION_RESULT_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_SUMMARY_EXCEL_PATH = Path("experiments_excel") / "experiment_summary_holdout.xlsx"


# ---------------------------------------------------------
# 1. Excel 값 파싱 유틸
# ---------------------------------------------------------

def _is_null_value(x):
    """Excel에서 읽힌 NaN/None/빈문자 처리."""
    if x is None:
        return True
    try:
        if pd.isna(x):
            return True
    except Exception:
        pass
    if isinstance(x, str) and x.strip().lower() in ["", "none", "nan", "null"]:
        return True
    return False


def parse_excel_object(value, default=None):
    """
    Excel cell에 저장된 list/dict/string 값을 Python object로 복원.
    예:
    "[1, 3, 5]" -> [1, 3, 5]
    "{'n_estimators': 500}" -> {"n_estimators": 500}
    """
    if _is_null_value(value):
        return default

    if isinstance(value, (list, dict, tuple)):
        return value

    if isinstance(value, (int, float, np.integer, np.floating)):
        return value

    text = str(value).strip()

    # Excel에서 dict/list가 문자열로 저장된 경우
    try:
        return ast.literal_eval(text)
    except Exception:
        pass

    try:
        return json.loads(text)
    except Exception:
        pass

    # "1,3,5,10" 형태 보정
    if "," in text and not any(ch in text for ch in ["{", "}", "[", "]"]):
        parts = [p.strip() for p in text.split(",") if p.strip() != ""]
        try:
            return [int(p) for p in parts]
        except Exception:
            return parts

    return text


def parse_int(value, default=None):
    if _is_null_value(value):
        return default
    return int(float(value))


def parse_float(value, default=None):
    if _is_null_value(value):
        return default
    return float(value)


def parse_str(value, default=None):
    if _is_null_value(value):
        return default
    return str(value)


def normalize_none(value):
    if _is_null_value(value):
        return None
    if isinstance(value, str) and value.strip().lower() in ["none", "nan", "null", ""]:
        return None
    return value


# ---------------------------------------------------------
# 2. Excel summary에서 config 가져오기
# ---------------------------------------------------------

def load_config_from_experiment_summary(
    summary_excel_path=DEFAULT_SUMMARY_EXCEL_PATH,
    row_idx=0,
    sheet_name="summary",
    sort_by=None,
    ascending=False,
    selection_mode="row",
    min_eval_count=10,
    min_pred_1_count=3,
    require_error_count_zero=True,
    verbose=True,
):
    """
    experiment_summary_holdout.xlsx에서 실험 config를 1개 가져온다.

    selection_mode
    --------------
    - "row": 기존 방식. Excel 현재 순서 또는 sort_by 정렬 후 row_idx 선택.
    - "best_precision": 최소 조건을 만족하는 후보 중 test_precision 높은 순 선택.
    - "best_balanced": 최소 조건을 만족하는 후보 중 precision/f1/pred_count를 같이 보는 점수로 선택.

    처음 테스트는 selection_mode="row", row_idx=0으로 쓰면 되고,
    실제 운영은 selection_mode="best_precision" 또는 "best_balanced" 추천.
    """
    summary_excel_path = Path(summary_excel_path)

    if not summary_excel_path.exists():
        raise FileNotFoundError(f"summary Excel 파일이 없습니다: {summary_excel_path}")

    summary_df = pd.read_excel(summary_excel_path, sheet_name=sheet_name)

    if len(summary_df) == 0:
        raise ValueError("summary Excel에 row가 없습니다.")

    work_df = summary_df.copy()
    work_df["_source_excel_row"] = np.arange(len(work_df))

    # 숫자형 컬럼 보정
    numeric_cols = [
        "test_precision", "test_recall", "test_f1", "test_auc", "test_logloss",
        "test_eval_count", "test_pred_1_count", "error_count",
        "n_days", "threshold", "top_n", "pred_threshold",
    ]
    for col in numeric_cols:
        if col in work_df.columns:
            work_df[col] = pd.to_numeric(work_df[col], errors="coerce")

    # -----------------------------------------------------
    # 1) 기존 테스트용: 0번 row 그대로 가져오기
    # -----------------------------------------------------
    if selection_mode == "row":
        if sort_by is not None:
            if sort_by not in work_df.columns:
                raise KeyError(f"sort_by 컬럼이 없습니다: {sort_by}")
            work_df = work_df.sort_values(sort_by, ascending=ascending).reset_index(drop=True)

        if row_idx >= len(work_df):
            raise IndexError(f"row_idx={row_idx}가 summary row 수({len(work_df)})보다 큽니다.")

        selected_df = work_df
        selected_reason = f"selection_mode=row, row_idx={row_idx}, sort_by={sort_by}"

    # -----------------------------------------------------
    # 2) 추천: 최소 샘플 조건 통과 후보 중 가장 높은 precision
    # -----------------------------------------------------
    elif selection_mode in ["best_precision", "best_balanced"]:
        filtered_df = work_df.copy()

        # -------------------------------------------------
        # 후보 필터링
        # - None인 조건은 필터를 걸지 않는다.
        # - Excel에서 숫자가 문자열로 들어와도 위에서 to_numeric 처리했기 때문에 비교 가능.
        # -------------------------------------------------
        filter_notes = []

        if require_error_count_zero and "error_count" in filtered_df.columns:
            before_n = len(filtered_df)
            filtered_df = filtered_df[filtered_df["error_count"].fillna(0) == 0].copy()
            filter_notes.append(f"error_count=0: {before_n}->{len(filtered_df)}")

        if min_eval_count is not None and "test_eval_count" in filtered_df.columns:
            before_n = len(filtered_df)
            filtered_df = filtered_df[filtered_df["test_eval_count"].fillna(0) >= min_eval_count].copy()
            filter_notes.append(f"test_eval_count>={min_eval_count}: {before_n}->{len(filtered_df)}")

        if min_pred_1_count is not None and "test_pred_1_count" in filtered_df.columns:
            before_n = len(filtered_df)
            filtered_df = filtered_df[filtered_df["test_pred_1_count"].fillna(0) >= min_pred_1_count].copy()
            filter_notes.append(f"test_pred_1_count>={min_pred_1_count}: {before_n}->{len(filtered_df)}")

        if "test_precision" not in work_df.columns:
            raise KeyError("best 선택에는 test_precision 컬럼이 필요합니다.")

        # 조건이 너무 빡세서 후보가 없으면 fallback
        if len(filtered_df) == 0:
            filtered_df = work_df.copy()
            selected_reason = (
                f"조건 통과 후보가 없어 전체에서 fallback 선택. "
                f"min_eval_count={min_eval_count}, min_pred_1_count={min_pred_1_count}, "
                f"require_error_count_zero={require_error_count_zero}. "
                f"filters={' | '.join(filter_notes) if filter_notes else '없음'}"
            )
        else:
            selected_reason = (
                f"조건 통과 후보 중 선택. "
                f"min_eval_count={min_eval_count}, min_pred_1_count={min_pred_1_count}, "
                f"require_error_count_zero={require_error_count_zero}. "
                f"filters={' | '.join(filter_notes) if filter_notes else '없음'}"
            )

        if selection_mode == "best_precision":
            sort_cols = [c for c in ["test_precision", "test_pred_1_count", "test_f1", "test_eval_count"] if c in filtered_df.columns]
            filtered_df = filtered_df.sort_values(sort_cols, ascending=[False] * len(sort_cols)).reset_index(drop=True)

        else:
            # precision만 보면 신호가 너무 적은 조합이 뽑힐 수 있어서 보정 점수 사용
            precision = filtered_df["test_precision"].fillna(0)
            f1 = filtered_df["test_f1"].fillna(0) if "test_f1" in filtered_df.columns else 0

            pred_count_base = min_pred_1_count if min_pred_1_count is not None else 1
            eval_count_base = min_eval_count if min_eval_count is not None else 1

            pred_count_factor = (
                np.minimum(1.0, filtered_df["test_pred_1_count"].fillna(0) / max(pred_count_base, 1))
                if "test_pred_1_count" in filtered_df.columns else 1.0
            )
            eval_count_factor = (
                np.minimum(1.0, filtered_df["test_eval_count"].fillna(0) / max(eval_count_base, 1))
                if "test_eval_count" in filtered_df.columns else 1.0
            )

            filtered_df["selection_score"] = (
                precision * 0.70
                + f1 * 0.30
            ) * pred_count_factor * eval_count_factor

            sort_cols = [c for c in ["selection_score", "test_precision", "test_pred_1_count", "test_f1"] if c in filtered_df.columns]
            filtered_df = filtered_df.sort_values(sort_cols, ascending=[False] * len(sort_cols)).reset_index(drop=True)

        if row_idx >= len(filtered_df):
            raise IndexError(f"row_idx={row_idx}가 후보 row 수({len(filtered_df)})보다 큽니다.")

        selected_df = filtered_df

    else:
        raise ValueError("selection_mode는 'row', 'best_precision', 'best_balanced' 중 하나여야 합니다.")

    row = selected_df.iloc[row_idx].to_dict()

    cfg = {
        "etf_code": parse_str(row.get("etf_code"), ETF_CODE),
        "n_days": parse_int(row.get("n_days"), N_DAYS),
        "threshold": parse_float(row.get("threshold"), THRESHOLD),

        # holdout Optuna 결과면 test_months가 있을 수 있음.
        # 모의투자에서는 직접 sim_start/sim_end를 쓰므로 필수는 아니지만 기록용으로 보존.
        "test_months": parse_int(row.get("test_months"), None),

        "vif_threshold": parse_float(row.get("vif_threshold"), VIF_THRESHOLD),
        "lag_search_years": parse_float(row.get("lag_search_years"), LAG_SEARCH_YEARS),
        "lag_days": parse_excel_object(row.get("lag_days"), LAG_DAYS),

        "random_state": parse_int(row.get("random_state"), RANDOM_STATE),
        "n_rf_runs": parse_int(row.get("n_rf_runs"), N_RF_RUNS),
        "n_repeats": parse_int(row.get("n_repeats"), N_REPEATS),
        "top_n": parse_int(row.get("top_n"), TOP_N),
        "pred_threshold": parse_float(row.get("pred_threshold"), PRED_THRESHOLD),

        "model_name": parse_str(row.get("model_name"), "random_forest"),
        "model_params": parse_excel_object(row.get("model_params"), {}),

        "start_date": normalize_none(row.get("start_date", START_DATE)),
        "end_date": normalize_none(row.get("end_date", END_DATE)),

        # 추적용
        "source_run_id": row.get("run_id", None),
        "source_experiment_key": row.get("experiment_key", None),
        "source_row_idx": int(row.get("_source_excel_row", row_idx)) if not _is_null_value(row.get("_source_excel_row", row_idx)) else row_idx,
        "selection_mode": selection_mode,
        "selection_reason": selected_reason,
    }

    if cfg["lag_days"] is None:
        cfg["lag_days"] = LAG_DAYS
    cfg["lag_days"] = [int(x) for x in cfg["lag_days"]]

    if cfg["model_params"] is None:
        cfg["model_params"] = {}

    if verbose:
        print("=" * 80)
        print("선택된 config")
        print("=" * 80)
        print("summary path:", summary_excel_path)
        print("selection_mode:", selection_mode)
        print("source_excel_row:", cfg["source_row_idx"])
        print("row_idx within selected candidates:", row_idx)
        print("sort_by:", sort_by)
        print("selection_reason:", selected_reason)
        if "test_precision" in row:
            print("source test_precision:", row.get("test_precision"))
        if "test_eval_count" in row:
            print("source test_eval_count:", row.get("test_eval_count"))
        if "test_pred_1_count" in row:
            print("source test_pred_1_count:", row.get("test_pred_1_count"))
        for k, v in cfg.items():
            print(f"{k}: {v}")
        print("=" * 80)

    return cfg, row, summary_df


# ---------------------------------------------------------
# 3. lag feature 생성: inference용
# ---------------------------------------------------------

def make_lagged_feature_frame_for_inference(
    df,
    best_lag_df,
    close_col,
    date_col="Date",
):
    """
    target 없이 inference용 lag feature frame을 만든다.
    make_lagged_dataset_by_best_lag()는 target까지 dropna하므로
    예측일 row를 만들 때는 별도 함수가 필요하다.
    """
    result = pd.DataFrame()
    result[date_col] = df[date_col].values
    result[close_col] = df[close_col].values

    lagged_feature_cols = []

    for _, row in best_lag_df.iterrows():
        feature = row["feature"]
        lag = int(row["lag"])

        if feature not in df.columns:
            continue

        lagged_col = f"{feature}_lag{lag}"
        result[lagged_col] = df[feature].shift(lag)
        lagged_feature_cols.append(lagged_col)

    result = result.replace([np.inf, -np.inf], np.nan)
    return result, lagged_feature_cols


# ---------------------------------------------------------
# 4. signal_date 기준 feature selection + train set 생성
# ---------------------------------------------------------

def build_feature_set_until_signal_date(
    base_df,
    base_feature_cols,
    close_col,
    signal_date,
    etf_code,
    n_days,
    threshold,
    vif_threshold=10,
    lag_search_years=1,
    lag_days=None,
    top_n=30,
    random_state=42,
    n_rf_runs=3,
    n_repeats=10,
    verbose=False,
):
    """
    signal_date까지의 데이터만 사용해 feature selection을 1회 수행한다.

    핵심:
    - base_df는 전체 데이터를 받아도 됨
    - 내부에서 Date <= signal_date까지만 잘라 사용
    - target은 train 내부에서만 생성
    - 마지막 n_days개는 미래 target을 모르므로 학습에서 자동 제외
    """
    if lag_days is None:
        lag_days = LAG_DAYS

    signal_date = pd.to_datetime(signal_date)

    base_df = base_df.copy().sort_values("Date").reset_index(drop=True)
    train_base_df = base_df[base_df["Date"] <= signal_date].copy().reset_index(drop=True)

    if len(train_base_df) < 250:
        raise ValueError(f"train 데이터가 너무 적습니다. signal_date={signal_date}, rows={len(train_base_df)}")

    if verbose:
        print("=" * 80)
        print("Feature selection 기준일:", signal_date)
        print("train_base_df 기간:", train_base_df["Date"].min(), "~", train_base_df["Date"].max())
        print("train_base_df shape:", train_base_df.shape)

    # 1. VIF는 signal_date까지의 train 데이터에서만 수행
    vif_feature_cols, removed_vif_df, final_vif_df = reduce_features_by_vif(
        df=train_base_df,
        feature_cols=base_feature_cols,
        vif_threshold=vif_threshold,
        date_col="Date",
        verbose=verbose,
    )

    vif_filtered_base_df = train_base_df[["Date", close_col] + vif_feature_cols].copy()

    # 2. target 생성
    target_df, target_col = add_target_column(
        df=vif_filtered_base_df,
        close_col=close_col,
        n_days=n_days,
        threshold=threshold,
    )

    # 실제로 학습 가능한 row만 사용
    target_df = target_df.dropna(subset=[target_col]).copy()
    target_df[target_col] = target_df[target_col].astype(int)

    if len(target_df) < 100:
        raise ValueError(f"target 학습 데이터가 너무 적습니다. rows={len(target_df)}")

    if target_df[target_col].nunique() < 2:
        raise ValueError(
            f"train target class가 1개뿐입니다. "
            f"target 분포: {target_df[target_col].value_counts().to_dict()}"
        )

    # 3. lag 탐색은 train target 데이터의 최근 lag_search_years 구간에서 수행
    max_target_date = target_df["Date"].max()
    lag_search_start = max_target_date - pd.DateOffset(years=lag_search_years)

    lag_search_df = target_df[target_df["Date"] >= lag_search_start].copy()

    if len(lag_search_df) < 100:
        # 최근 구간이 너무 짧으면 전체 target_df 사용
        lag_search_df = target_df.copy()

    lag_result_df, best_lag_df = find_best_lag_by_feature(
        df=lag_search_df,
        feature_cols=vif_feature_cols,
        target_col=target_col,
        lag_days=lag_days,
        date_col="Date",
    )

    # 4. best lag 적용한 train lagged dataset 생성
    lagged_df, lagged_feature_cols = make_lagged_dataset_by_best_lag(
        df=target_df,
        best_lag_df=best_lag_df,
        target_col=target_col,
        close_col=close_col,
        n_days=n_days,
        date_col="Date",
    )

    if len(lagged_df) < 100:
        raise ValueError(f"lagged_df가 너무 적습니다. rows={len(lagged_df)}")

    if len(lagged_feature_cols) == 0:
        raise ValueError("lagged_feature_cols가 비어 있습니다.")

    # 5. RF permutation importance로 top feature 선택
    importance_df, raw_importance_df, baseline_df = run_rf_permutation_importance_in_sample(
        lagged_df=lagged_df,
        feature_cols=lagged_feature_cols,
        target_col=target_col,
        date_col="Date",
        close_col=close_col,
        n_rf_runs=n_rf_runs,
        n_repeats=n_repeats,
        random_state=random_state,
    )

    importance_view_df = importance_df.copy()
    importance_view_df[["base_feature", "selected_lag"]] = importance_view_df["feature"].apply(
        lambda x: pd.Series(split_lagged_feature_name(x))
    )

    importance_with_lag_df = importance_view_df.merge(
        best_lag_df.rename(columns={
            "feature": "base_feature",
            "lag": "best_lag",
            "corr": "lag_corr",
            "abs_corr": "lag_abs_corr",
            "n_rows": "lag_n_rows",
        }),
        on="base_feature",
        how="left",
    )

    top_feature_df = importance_with_lag_df.sort_values(
        "importance_score",
        ascending=False,
    ).head(top_n).copy().reset_index(drop=True)

    top_feature_cols = top_feature_df["feature"].tolist()

    if len(top_feature_cols) == 0:
        raise ValueError("top_feature_cols가 비어 있습니다.")

    result = {
        "signal_date": signal_date,
        "train_base_df": train_base_df,
        "vif_filtered_base_df": vif_filtered_base_df,
        "vif_feature_cols": vif_feature_cols,
        "removed_vif_df": removed_vif_df,
        "final_vif_df": final_vif_df,
        "target_df": target_df,
        "target_col": target_col,
        "lag_result_df": lag_result_df,
        "best_lag_df": best_lag_df,
        "lagged_df": lagged_df,
        "lagged_feature_cols": lagged_feature_cols,
        "importance_df": importance_df,
        "raw_importance_df": raw_importance_df,
        "baseline_df": baseline_df,
        "importance_with_lag_df": importance_with_lag_df,
        "top_feature_df": top_feature_df,
        "top_feature_cols": top_feature_cols,
    }

    return result


# ---------------------------------------------------------
# 5. 실제 n_days 뒤 결과 계산
# ---------------------------------------------------------

def get_actual_future_result(
    base_df,
    signal_date,
    close_col,
    n_days,
    threshold,
):
    """
    signal_date 기준 n_days 뒤 실제 수익률/target 계산.
    base_df 전체 데이터에서 조회하므로, 역사적 모의투자 평가용이다.
    """
    df = base_df.copy().sort_values("Date").reset_index(drop=True)
    signal_date = pd.to_datetime(signal_date)

    matches = df.index[df["Date"] == signal_date].tolist()
    if len(matches) == 0:
        return {
            "actual_available": False,
            "future_date": pd.NaT,
            "signal_close": np.nan,
            "future_close": np.nan,
            "future_ret": np.nan,
            "actual_target": np.nan,
        }

    idx = matches[0]
    future_idx = idx + int(n_days)

    signal_close = df.loc[idx, close_col]

    if future_idx >= len(df):
        return {
            "actual_available": False,
            "future_date": pd.NaT,
            "signal_close": signal_close,
            "future_close": np.nan,
            "future_ret": np.nan,
            "actual_target": np.nan,
        }

    future_date = df.loc[future_idx, "Date"]
    future_close = df.loc[future_idx, close_col]
    future_ret = future_close / signal_close - 1
    actual_target = int(future_ret >= threshold)

    return {
        "actual_available": True,
        "future_date": future_date,
        "signal_close": signal_close,
        "future_close": future_close,
        "future_ret": future_ret,
        "actual_target": actual_target,
    }


# ---------------------------------------------------------
# 6. 기준일 1개 예측
# ---------------------------------------------------------

def predict_one_action_date(
    base_df,
    base_feature_cols,
    close_col,
    action_date,
    cfg,
    verbose=False,
):
    """
    action_date 기준 모의투자 예측 1회.

    처리 기준:
    - action_date의 전 거래일을 signal_date로 사용
    - signal_date까지의 데이터만 사용해 feature selection + train + prediction
    - 실제 결과는 signal_date 기준 n_days 뒤 수익률로 비교
    """
    base_df = base_df.copy().sort_values("Date").reset_index(drop=True)
    action_date = pd.to_datetime(action_date)

    trading_dates = base_df["Date"].tolist()
    prev_dates = [d for d in trading_dates if d < action_date]

    if len(prev_dates) == 0:
        raise ValueError(f"action_date 이전 거래일이 없습니다: {action_date}")

    signal_date = max(prev_dates)

    n_days = int(cfg["n_days"])
    threshold = float(cfg["threshold"])
    pred_threshold = float(cfg.get("pred_threshold", PRED_THRESHOLD))
    random_state = int(cfg.get("random_state", RANDOM_STATE))
    model_name = cfg.get("model_name", "random_forest")
    model_params = cfg.get("model_params", {}) or {}

    # 1. signal_date까지 feature selection
    feature_result = build_feature_set_until_signal_date(
        base_df=base_df,
        base_feature_cols=base_feature_cols,
        close_col=close_col,
        signal_date=signal_date,
        etf_code=cfg.get("etf_code", ETF_CODE),
        n_days=n_days,
        threshold=threshold,
        vif_threshold=float(cfg.get("vif_threshold", VIF_THRESHOLD)),
        lag_search_years=float(cfg.get("lag_search_years", LAG_SEARCH_YEARS)),
        lag_days=cfg.get("lag_days", LAG_DAYS),
        top_n=int(cfg.get("top_n", TOP_N)),
        random_state=random_state,
        n_rf_runs=int(cfg.get("n_rf_runs", N_RF_RUNS)),
        n_repeats=int(cfg.get("n_repeats", N_REPEATS)),
        verbose=verbose,
    )

    top_feature_cols = feature_result["top_feature_cols"]
    target_col = feature_result["target_col"]
    lagged_df = feature_result["lagged_df"]

    # 2. train set 준비
    train_model_df = lagged_df[["Date", target_col] + top_feature_cols].copy()
    train_model_df = train_model_df.replace([np.inf, -np.inf], np.nan).dropna().copy()

    if train_model_df[target_col].nunique() < 2:
        raise ValueError(f"학습 target class가 1개뿐입니다: {train_model_df[target_col].value_counts().to_dict()}")

    X_train = train_model_df[top_feature_cols].copy()
    y_train = train_model_df[target_col].astype(int).copy()

    # 3. 모델 학습
    model = make_classifier_model(
        model_name=model_name,
        random_state=random_state,
        model_params=model_params,
    )
    model.fit(X_train, y_train)

    # 4. signal_date inference row 생성
    inference_df, inference_feature_cols = make_lagged_feature_frame_for_inference(
        df=feature_result["vif_filtered_base_df"],
        best_lag_df=feature_result["best_lag_df"],
        close_col=close_col,
        date_col="Date",
    )

    pred_row = inference_df[inference_df["Date"] == signal_date].copy()

    if len(pred_row) == 0:
        raise ValueError(f"signal_date의 inference row가 없습니다: {signal_date}")

    pred_row = pred_row.tail(1).copy()

    missing_cols = [col for col in top_feature_cols if col not in pred_row.columns]
    if len(missing_cols) > 0:
        raise ValueError(f"inference row에 없는 feature가 있습니다: {missing_cols}")

    X_pred = pred_row[top_feature_cols].replace([np.inf, -np.inf], np.nan)

    if X_pred.isna().any(axis=None):
        nan_cols = X_pred.columns[X_pred.isna().any()].tolist()
        raise ValueError(f"inference feature에 NA가 있습니다: {nan_cols}")

    # 5. 예측 확률/신호
    if hasattr(model, "predict_proba"):
        pred_proba = float(model.predict_proba(X_pred)[:, 1][0])
    else:
        # 대부분의 현재 모델은 predict_proba 가능하지만 안전장치
        pred_proba = float(model.predict(X_pred)[0])

    pred = int(pred_proba >= pred_threshold)

    # 6. 실제 결과 계산
    actual = get_actual_future_result(
        base_df=base_df,
        signal_date=signal_date,
        close_col=close_col,
        n_days=n_days,
        threshold=threshold,
    )

    selected_feature_list = " | ".join(top_feature_cols)
    selected_feature_top10 = " | ".join(top_feature_cols[:10])

    record = {
        "action_date": action_date,
        "signal_date": signal_date,
        "future_date": actual["future_date"],

        "etf_code": cfg.get("etf_code", ETF_CODE),
        "n_days": n_days,
        "threshold": threshold,
        "pred_threshold": pred_threshold,
        "model_name": model_name,
        "model_params": str(model_params),

        "pred_proba": pred_proba,
        "pred": pred,

        "actual_available": actual["actual_available"],
        "actual_target": actual["actual_target"],
        "future_ret": actual["future_ret"],
        "signal_close": actual["signal_close"],
        "future_close": actual["future_close"],

        "is_correct": (
            int(pred == actual["actual_target"])
            if actual["actual_available"] and not pd.isna(actual["actual_target"])
            else np.nan
        ),

        "selected_feature_count": len(top_feature_cols),
        "selected_feature_list": selected_feature_list,
        "selected_feature_top10": selected_feature_top10,

        "train_row_count": len(train_model_df),
        "train_target_1_count": int((y_train == 1).sum()),
        "train_target_0_count": int((y_train == 0).sum()),

        "source_run_id": cfg.get("source_run_id", None),
        "source_experiment_key": cfg.get("source_experiment_key", None),
    }

    return record, feature_result, model


# ---------------------------------------------------------
# 7. 지정 기간 rolling 모의투자 실행
# ---------------------------------------------------------



# ---------------------------------------------------------
# 4. 시뮬레이션 결과 그래프
# ---------------------------------------------------------

def make_simulation_pred_actual_plot(result_df, title="Simulation Result: Pred vs Actual Target"):
    """
    simulation_result["result_df"] 기준으로
    x축 action_date, y축 pred / actual_target 그래프를 생성한다.
    """
    if result_df is None or len(result_df) == 0:
        raise ValueError("result_df가 비어 있어서 그래프를 만들 수 없습니다.")

    plot_df = result_df.copy()
    plot_df["action_date"] = pd.to_datetime(plot_df["action_date"])
    plot_df = plot_df.sort_values("action_date").reset_index(drop=True)

    fig = go.Figure()

    # 실제 target
    fig.add_trace(
        go.Scatter(
            x=plot_df["action_date"],
            y=plot_df["actual_target"],
            mode="lines+markers",
            name="Actual Target",
            line=dict(width=2),
            marker=dict(size=8),
            customdata=np.stack(
                [
                    plot_df.get("future_ret", pd.Series([np.nan] * len(plot_df))),
                    plot_df.get("actual_available", pd.Series([np.nan] * len(plot_df))),
                ],
                axis=-1,
            ),
            hovertemplate=(
                "Action Date: %{x|%Y-%m-%d}<br>"
                "Actual Target: %{y}<br>"
                "Future Ret: %{customdata[0]:.2%}<br>"
                "Actual Available: %{customdata[1]}<extra></extra>"
            ),
        )
    )

    # 예측값 pred: actual과 겹쳐 보이는 것을 줄이려고 0.04만 위로 띄워서 표시
    fig.add_trace(
        go.Scatter(
            x=plot_df["action_date"],
            y=plot_df["pred"] + 0.04,
            mode="lines+markers",
            name="Pred",
            line=dict(width=2, dash="dot"),
            marker=dict(size=8, symbol="x"),
            customdata=np.stack(
                [
                    plot_df.get("pred", pd.Series([np.nan] * len(plot_df))),
                    plot_df.get("pred_proba", pd.Series([np.nan] * len(plot_df))),
                    plot_df.get("actual_target", pd.Series([np.nan] * len(plot_df))),
                    plot_df.get("future_ret", pd.Series([np.nan] * len(plot_df))),
                ],
                axis=-1,
            ),
            hovertemplate=(
                "Action Date: %{x|%Y-%m-%d}<br>"
                "Pred: %{customdata[0]}<br>"
                "Pred Proba: %{customdata[1]:.4f}<br>"
                "Actual Target: %{customdata[2]}<br>"
                "Future Ret: %{customdata[3]:.2%}<extra></extra>"
            ),
        )
    )

    fig.update_layout(
        title=title,
        xaxis_title="Action Date",
        yaxis_title="Pred / Actual Target",
        height=520,
        width=1150,
        hovermode="x unified",
        yaxis=dict(
            tickmode="array",
            tickvals=[0, 1],
            ticktext=["0", "1"],
            range=[-0.15, 1.2],
        ),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1,
        ),
    )

    return fig


def summarize_simulation_result(result_df):
    """모의투자 예측 결과 요약."""
    if result_df is None or len(result_df) == 0:
        return pd.DataFrame()

    eval_df = result_df[result_df["actual_available"] == True].copy()
    eval_df = eval_df.dropna(subset=["actual_target", "pred"]).copy()

    if len(eval_df) == 0:
        return pd.DataFrame([{
            "eval_count": 0,
            "accuracy": np.nan,
            "precision": np.nan,
            "recall": np.nan,
            "f1": np.nan,
            "pred_1_count": 0,
            "actual_1_count": 0,
            "avg_future_ret_when_pred_1": np.nan,
            "strategy_compound_return": np.nan,
        }])

    y_true = eval_df["actual_target"].astype(int)
    y_pred = eval_df["pred"].astype(int)

    metrics = safe_binary_metrics(
        y_true=y_true,
        pred=y_pred,
        pred_proba=eval_df["pred_proba"] if "pred_proba" in eval_df.columns else None,
    )

    pred_1_df = eval_df[eval_df["pred"] == 1].copy()

    # 단순 모의투자 수익률: pred=1이면 future_ret만큼 투자, pred=0이면 0% 수익으로 가정
    # n_days가 겹치는 경우 실제 포트폴리오 수익률과는 다를 수 있음.
    strategy_period_ret = np.where(eval_df["pred"] == 1, eval_df["future_ret"], 0.0)
    strategy_compound_return = float(np.prod(1 + strategy_period_ret) - 1)

    summary = {
        "eval_count": len(eval_df),
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "auc": metrics["auc"],
        "logloss": metrics["logloss"],

        "tn": metrics["tn"],
        "fp": metrics["fp"],
        "fn": metrics["fn"],
        "tp": metrics["tp"],

        "pred_1_count": int((y_pred == 1).sum()),
        "actual_1_count": int((y_true == 1).sum()),

        "avg_future_ret_all": float(eval_df["future_ret"].mean()),
        "avg_future_ret_when_pred_1": (
            float(pred_1_df["future_ret"].mean()) if len(pred_1_df) > 0 else np.nan
        ),
        "strategy_compound_return": strategy_compound_return,
    }

    return pd.DataFrame([summary])


def run_paper_trading_simulation(
    summary_excel_path=DEFAULT_SUMMARY_EXCEL_PATH,
    summary_row_idx=0,
    sort_by=None,
    selection_mode="row",
    min_eval_count=10,
    min_pred_1_count=3,
    require_error_count_zero=True,
    sim_start_date=None,
    sim_end_date=None,
    max_action_dates=None,
    save_result=True,
    result_dir=SIMULATION_RESULT_DIR,
    verbose=True,
    feature_verbose=False,
):
    """
    Excel summary에서 config를 읽어 지정 기간 동안 rolling 모의투자 실행.

    Parameters
    ----------
    summary_row_idx : int
        selection_mode="row"일 때는 Excel row 번호.
        best_precision/best_balanced일 때는 후보 정렬 후 몇 번째를 쓸지.
    sort_by : str or None
        selection_mode="row"에서만 주로 사용.
    selection_mode : str
        "row"는 테스트용 0번 row 그대로 사용.
        "best_precision"은 최소 조건 통과 후보 중 test_precision 최고 조합 사용.
        "best_balanced"는 precision + f1 + 신호 수를 같이 고려한 score로 선택.
    sim_start_date, sim_end_date : str
        모의투자 action_date 기간.
    max_action_dates : int or None
        테스트용으로 앞에서 몇 개 action_date만 실행할지 제한.
        처음에는 3~5 정도 추천.
    """
    result_dir = Path(result_dir)
    result_dir.mkdir(parents=True, exist_ok=True)

    # 1. config 로드
    cfg, source_row, summary_df = load_config_from_experiment_summary(
        summary_excel_path=summary_excel_path,
        row_idx=summary_row_idx,
        sort_by=sort_by,
        ascending=False,
        selection_mode=selection_mode,
        min_eval_count=min_eval_count,
        min_pred_1_count=min_pred_1_count,
        require_error_count_zero=require_error_count_zero,
        verbose=verbose,
    )

    etf_code = cfg["etf_code"]

    # 2. 전체 base_df 생성
    # 실제 결과 비교를 위해 sim_end_date 이후 n_days 데이터까지 필요하므로 end_date는 cfg end_date를 그대로 사용
    if verbose:
        print("\nBase feature dataset 생성 중...")

    base_df, base_feature_cols, close_col = make_base_feature_dataset(
        etf_code=etf_code,
        external_tickers=EXTERNAL_TICKERS,
        external_feature_types=EXTERNAL_FEATURE_TYPES,
        start_date=cfg.get("start_date") or START_DATE,
        end_date=cfg.get("end_date") or END_DATE,
    )

    base_df = base_df.sort_values("Date").reset_index(drop=True)

    if sim_start_date is None:
        # 기본값: 최근 1개월
        sim_start_date = base_df["Date"].max() - pd.DateOffset(months=1)
    else:
        sim_start_date = pd.to_datetime(sim_start_date)

    if sim_end_date is None:
        sim_end_date = base_df["Date"].max()
    else:
        sim_end_date = pd.to_datetime(sim_end_date)

    # 3. action date 목록
    action_dates = base_df.loc[
        (base_df["Date"] >= sim_start_date) &
        (base_df["Date"] <= sim_end_date),
        "Date"
    ].tolist()

    if max_action_dates is not None:
        action_dates = action_dates[:int(max_action_dates)]

    if len(action_dates) == 0:
        raise ValueError(f"sim 기간에 해당하는 거래일이 없습니다: {sim_start_date} ~ {sim_end_date}")

    if verbose:
        print("=" * 80)
        print("모의투자 실행 기간")
        print("sim_start_date:", sim_start_date)
        print("sim_end_date:", sim_end_date)
        print("action date 수:", len(action_dates))
        print("첫 action_date:", action_dates[0])
        print("마지막 action_date:", action_dates[-1])
        print("=" * 80)

    # 4. rolling simulation
    records = []
    errors = []

    for i, action_date in enumerate(action_dates, start=1):
        if verbose:
            print("\n" + "=" * 80)
            print(f"[{i}/{len(action_dates)}] action_date={pd.to_datetime(action_date).date()}")
            print("=" * 80)

        try:
            record, feature_result, model = predict_one_action_date(
                base_df=base_df,
                base_feature_cols=base_feature_cols,
                close_col=close_col,
                action_date=action_date,
                cfg=cfg,
                verbose=feature_verbose,
            )

            records.append(record)

            if verbose:
                print(
                    f"signal_date={pd.to_datetime(record['signal_date']).date()} | "
                    f"proba={record['pred_proba']:.4f} | "
                    f"pred={record['pred']} | "
                    f"actual={record['actual_target']} | "
                    f"future_ret={record['future_ret']}"
                )

        except Exception as e:
            error_record = {
                "action_date": action_date,
                "error": str(e),
            }
            errors.append(error_record)

            if verbose:
                print("[ERROR]", error_record)

    result_df = pd.DataFrame(records)
    error_df = pd.DataFrame(errors)
    summary_result_df = summarize_simulation_result(result_df)

    # 5. 결과 그래프 생성
    plot_title = f"{etf_code} Paper Trading Simulation: Pred vs Actual Target"
    fig = None
    if result_df is not None and len(result_df) > 0:
        fig = make_simulation_pred_actual_plot(result_df, title=plot_title)

    # 6. 결과 저장
    save_info = {}

    if save_result:
        # 파일명 기준: 시작일-종료일-timestamp
        # 예: 20260201-20260430-20260526_172300_summary.csv
        period_start_str = pd.to_datetime(sim_start_date).strftime("%Y%m%d")
        period_end_str = pd.to_datetime(sim_end_date).strftime("%Y%m%d")
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        file_prefix = f"{period_start_str}-{period_end_str}-{timestamp}"

        xlsx_path = result_dir / f"{file_prefix}_paper_trading_simulation_{etf_code}.xlsx"
        summary_csv_path = result_dir / f"{file_prefix}_summary.csv"
        result_csv_path = result_dir / f"{file_prefix}_result.csv"
        error_csv_path = result_dir / f"{file_prefix}_errors.csv"
        graph_html_path = result_dir / f"{file_prefix}_pred_actual_graph.html"

        # Excel 통합 저장
        with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
            pd.DataFrame([cfg]).to_excel(writer, index=False, sheet_name="config")
            summary_result_df.to_excel(writer, index=False, sheet_name="summary")
            result_df.to_excel(writer, index=False, sheet_name="result")
            error_df.to_excel(writer, index=False, sheet_name="errors")

        # 사용자가 display하던 두 개를 각각 CSV로도 저장
        summary_result_df.to_csv(summary_csv_path, index=False, encoding="utf-8-sig")
        result_df.to_csv(result_csv_path, index=False, encoding="utf-8-sig")
        error_df.to_csv(error_csv_path, index=False, encoding="utf-8-sig")

        # pred / actual_target 그래프 저장
        if fig is not None:
            fig.write_html(graph_html_path, include_plotlyjs="cdn")

        save_info = {
            "file_prefix": file_prefix,
            "xlsx_path": str(xlsx_path),
            "summary_csv_path": str(summary_csv_path),
            "result_csv_path": str(result_csv_path),
            "error_csv_path": str(error_csv_path),
            "graph_html_path": str(graph_html_path) if fig is not None else None,
            # 기존 코드 호환용
            "output_path": str(xlsx_path),
        }

        if verbose:
            print("\n저장 완료")
            print("Excel     :", xlsx_path)
            print("SummaryCSV:", summary_csv_path)
            print("ResultCSV :", result_csv_path)
            print("GraphHTML :", graph_html_path if fig is not None else None)

    return {
        "cfg": cfg,
        "source_row": source_row,
        "base_df": base_df,
        "base_feature_cols": base_feature_cols,
        "close_col": close_col,
        "result_df": result_df,
        "summary_df": summary_result_df,
        "error_df": error_df,
        "fig": fig,
        "save_info": save_info,
    }


## 실행 예시


In [ ]:

# =========================================================
# 실행 예시
# =========================================================

# Optuna 결과 Excel 경로
SUMMARY_EXCEL_PATH = Path("experiments_excel") / "experiment_summary_holdout.xlsx"

# ---------------------------------------------------------
# 1) 테스트용: 기존처럼 Excel 0번 row 그대로 사용
# ---------------------------------------------------------
# 처음에는 max_action_dates=3 정도로 짧게 테스트 추천

# simulation_result = run_paper_trading_simulation(
#     summary_excel_path=SUMMARY_EXCEL_PATH,
#     summary_row_idx=0,
#     selection_mode="row",      # 테스트용: Excel 0번 row 그대로 사용
#     sort_by=None,
#     sim_start_date="2026-02-01",
#     sim_end_date="2026-04-30",
#     max_action_dates=3,
#     save_result=True,
#     verbose=True,
#     feature_verbose=False,
# )

# print("저장 경로:", simulation_result["save_info"].get("output_path"))
# display(simulation_result["summary_df"])
# display(simulation_result["result_df"].head())


# ---------------------------------------------------------
# 2) 실사용 추천: 최소 조건 통과 후보 중 최고 precision 조합 사용
# ---------------------------------------------------------
simulation_result = run_paper_trading_simulation(
    summary_excel_path=SUMMARY_EXCEL_PATH,
    summary_row_idx=0,
    selection_mode="best_precision",  # 추천: test_precision 최고 조합 자동 선택
    min_eval_count=50,
    min_pred_1_count=15,
    require_error_count_zero=True,
    sim_start_date="2026-02-01",
    sim_end_date="2026-04-30",
    max_action_dates=50,          # None이면 전체기간 실행, 3이면 sim_start_date 기준 3일만 실행
    save_result=True,
    verbose=True,
    feature_verbose=False,
)

print("저장 경로:", simulation_result["save_info"].get("output_path"))
print("summary csv:", simulation_result["save_info"].get("summary_csv_path"))
print("result csv:", simulation_result["save_info"].get("result_csv_path"))
print("graph html:", simulation_result["save_info"].get("graph_html_path"))

display(simulation_result["summary_df"])
display(simulation_result["result_df"])

if simulation_result.get("fig") is not None:
    simulation_result["fig"].show()


# ---------------------------------------------------------
# 3) 더 보수적 추천: precision만 보지 않고 f1/신호 수까지 보정
# ---------------------------------------------------------
# simulation_result = run_paper_trading_simulation(
#     summary_excel_path=SUMMARY_EXCEL_PATH,
#     summary_row_idx=0,
#     selection_mode="best_balanced",
#     min_eval_count=30,
#     min_pred_1_count=3,
#     require_error_count_zero=True,
#     sim_start_date="2026-02-01",
#     sim_end_date="2026-04-30",
#     max_action_dates=None,
#     save_result=True,
#     verbose=True,
#     feature_verbose=False,
# )

print("저장 경로:", simulation_result["save_info"].get("output_path"))
print("summary csv:", simulation_result["save_info"].get("summary_csv_path"))
print("result csv:", simulation_result["save_info"].get("result_csv_path"))
print("graph html:", simulation_result["save_info"].get("graph_html_path"))

display(simulation_result["summary_df"])
display(simulation_result["result_df"])

if simulation_result.get("fig") is not None:
    simulation_result["fig"].show()



선택된 config
summary path: experiments_excel\experiment_summary_holdout.xlsx
selection_mode: best_precision
source_excel_row: 68
row_idx within selected candidates: 0
sort_by: None
selection_reason: 조건 통과 후보 중 선택. min_eval_count=50, min_pred_1_count=15, require_error_count_zero=True. filters=error_count=0: 189->189 | test_eval_count>=50: 189->76 | test_pred_1_count>=15: 76->27
source test_precision: 0.7931034482758621
source test_eval_count: 61
source test_pred_1_count: 29
etf_code: SMH
n_days: 3
threshold: 0.005
test_months: 3
vif_threshold: 10.0
lag_search_years: 1.0
lag_days: [1, 3, 5, 10, 20, 40, 60, 120]
random_state: 42
n_rf_runs: 3
n_repeats: 10
top_n: 30
pred_threshold: 0.45
model_name: gradient_boosting
model_params: {'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 5}
start_date: 2020-01-01
end_date: None
source_run_id: 20260523_204527_optuna_holdout_trial_0002
source_experiment_key: 2712411cbd55db28d1fff9447685cffe
source_row_idx: 68
selection_mode: best_precision
selec